# DINGO: real-source workflow demo

Run these cells in order with the **grism** kernel. The main example is **ID3700**: inspect imaging → inspect candidate residuals → choose a start → run a short MCMC → view and save results.

By default this reuses ID3700's current accepted imaging fit, creates a matching candidate search, then runs **1,000 warmup + 1,000 retained steps per walker**. This is a pilot for learning the workflow; it does not establish reportable errors. Progress and checkpoints are automatic. The candidate-selection cell asks which solution you want to start from.

The model is the original three-file joint R+C fit with **intrinsic dispersion fixed to zero**. It samples deprojected **$V_\infty$** and inclination, using a **5° Gaussian inclination prior width** by default. Each source/experiment has its own handle; there is no global source ID.

For viewing a completed run only, execute **Setup**, then the final **Reopen saved results** cell. [Full API guide](docs/workflow.md).

## Setup

Change `WORKING_DIR` to choose where all new source/run folders are saved. Existing observations and historical results stay in their current locations.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
from dingo import workflow as wf

KINEMATICS_DIR = Path(wf.__file__).resolve().parents[2]
WORKING_DIR = KINEMATICS_DIR / "working_results"  # Change this save location.
PREVIOUS_RESULTS = KINEMATICS_DIR / "results"

print("Save location:", WORKING_DIR)
print("API:", Path(wf.__file__).resolve())

## 1. Prepare ID3700

`reuse_imaging_3700=True` reads the current accepted PySersic fit; set it to `False` to fit imaging afresh. A new candidate search is made inside this run, then reused on later calls. ID3700's historical search refers to older imaging files, so it is not the default parent here.

`inclination_sigma_deg=5.0` is adjustable per run. `overwrite=False` resumes/reopens this exact experiment. `overwrite=True` deletes **only this named run**, including its local checkpoints and plots; it preserves external imaging. Use a different `run_number` to compare experiments. If you change the imaging choice after preparing a search, also choose a new run number or explicitly overwrite.

In [ ]:
reuse_imaging_3700 = True

source3700 = wf.prepare_source(
    3700,
    working_dir=WORKING_DIR,
    run_number=1,
    inclination_mode="imaging_prior",
    inclination_sigma_deg=5.0,  # Total Gaussian 1σ width, in degrees.
    parent_search=None,       # Create/reuse a matching search inside this run.
    device="cuda",
    overwrite=False,
    notes="Demo: imaging inclination prior; candidate chosen from residual inspection.",
)
print("This run:", source3700.directory)

## 2. Inspect the imaging first

This shows the actual F150W fitting image, fit, **native PySersic corner plot**, and derived inclination for both `q0=0` and `q0=0.2`. PySersic fits axis ratio; the inclination is derived. The grism prior uses the existing full/core imaging-mixture mean with the total width chosen above. It does not force the gas morphology or center to follow the stars.

In [ ]:
if reuse_imaging_3700:
    imaging3700 = wf.use_imaging(
        source3700,
        PREVIOUS_RESULTS / "real_sources/ID3700/imaging_runs/pysersic_v1",
    )
else:
    imaging3700 = wf.fit_imaging(source3700, warmup=1000, draws=2000)

_ = wf.show_imaging(source3700)

## 3. Inspect every candidate

The table and residual panels use the same original candidate IDs. Look for coherent leftover structure and compare velocities. A search row is a possible start, not a marginalized posterior or proof of a unique solution. Fresh searches can take appreciable time; progress appears below.

In [ ]:
candidates3700 = wf.find_candidates(source3700)
_ = wf.show_candidates(source3700)

## 4. Choose and preview the exact start

Enter a **candidate ID from the table**. This records the choice for this run. A different candidate needs a new run number or an explicit overwrite; rerunning this cell with the same choice is fine.

In [ ]:
candidate3700 = int(input("ID3700 candidate ID to initialize from: "))
start3700 = wf.prepare_start(source3700, candidate_id=candidate3700)
figure = start3700.figure()
display(figure)
plt.close(figure)

## 5. Run MCMC

MCMC is enabled. Set `RUN_MCMC_3700=False` to load this run's existing posterior. `draws` is the **total retained target per walker**, excluding warmup. To extend this pilot, increase only `draws` (for example to `8000`); keep warmup, walkers and seed unchanged. Changing those settings requires another run.

The PyTorch likelihood runs on CUDA; emcee supplies the ensemble proposals. Walkers interact, so they are not independent chains. The directory and live progress are printed below.

In [ ]:
RUN_MCMC_3700 = True

if RUN_MCMC_3700:
    posterior3700 = wf.fit_source(
        source3700,
        warmup=1000,
        draws=1000,  # Pilot; increase this total to extend the same run.
        walkers=64,
        seed=42,
        notes="Pilot run; inspect residuals and every walker before interpreting intervals.",
    )
else:
    posterior3700 = wf.load_results(source3700)

print(posterior3700.status)

## 6. Inspect the posterior

Residuals use the **highest-posterior saved draw**. The monochrome corner and velocity table use **pooled medians and 16th–84th percentiles**. They can differ when walkers occupy different solutions. Every walker is shown; none is silently removed. Here $V_\infty$ is the asymptotic amplitude, and the table also identifies the finite radius used for its velocity comparison.

In [ ]:
_ = wf.show_results(source3700)

## 7. Save the final files

This writes `results.npz`, plots and `run_info.txt` into the printed source/run folder. Sampler details stay in `_checkpoints`. Saving records the run; it does not certify convergence.

Mass profiles can be added with `wf.save_results(source3700, radii_pc=observed_radii_pc)` after choosing radii supported by the observations. No mass plot is exported by default.

In [ ]:
saved3700 = wf.save_results(source3700)
print("Saved files:", saved3700)
print("Run information:", saved3700 / "run_info.txt")
print("Posterior:", saved3700 / "results.npz")

## Optional: a second source, with a manual start

This separate **ID15665** example starts at $V_\infty=400$ km/s and **fixes inclination at 30°**. These are test choices, not a claim that this is the preferred solution. The existing source search supplies the other starting coordinates. The exact fixed-angle start is displayed before sampling; its uncertainty excludes inclination uncertainty.

Set `RUN_MANUAL_15665=True` to run it. The ID3700 handle remains available. To sample inclination instead, use `inclination_mode="imaging_prior"`, `inclination_deg=None`, and the desired matching prior width; `start_inclination_deg` is only a starting value. Use a new run name/number when changing the experiment.

In [ ]:
RUN_MANUAL_15665 = False

if RUN_MANUAL_15665:
    source15665 = wf.prepare_source(
        15665,
        working_dir=WORKING_DIR,
        run_number=1,
        inclination_mode="fixed",
        inclination_deg=30.0,
        inclination_sigma_deg=5.0,  # Ignored while inclination is fixed.
        start_velocity_kms=400.0,
        start_inclination_deg=30.0,
        parent_search=PREVIOUS_RESULTS / "three_file_sources/ID15665/rotation_v_i5_v1",
        device="cuda",
        overwrite=False,
        notes="Manual-start sensitivity experiment: Vinf=400 km/s, exact i=30 degrees.",
    )
    # With no explicit candidate ID, use the highest-posterior finite parent
    # candidate for the other initial coordinates; the choice is recorded.
    start15665 = wf.prepare_start(source15665)
    figure = start15665.figure()
    display(figure)
    plt.close(figure)

    posterior15665 = wf.fit_source(
        source15665, warmup=1000, draws=1000, walkers=64, seed=42,
    )
    _ = wf.show_results(source15665)
    saved15665 = wf.save_results(source15665)
    print("ID15665 files:", saved15665)
else:
    print("Optional ID15665 experiment is disabled.")

Once you have prepared and inspected several starts, you can also run them sequentially with:

```python
outcomes = wf.fit_batch(
    {"3700": source3700, "15665 fixed30": source15665},
    warmup=1000, draws=8000, walkers=64, seed=42,
)
```

Each entry retains its own settings and files. Review and save each successful result separately. See the [API guide](docs/workflow.md) for failure handling and other inclination choices.

## Reopen saved results — no fitting

After a kernel restart, run **Setup** and this cell only. Paste the **complete run directory** printed when saving. Leaving it as `None` does nothing. This opens that exact run; it never silently picks a latest run or starts MCMC.

In [ ]:
saved_run_directory = None  # e.g. WORKING_DIR / "3700/3700_priorinc_s5_search_run01"

if saved_run_directory is not None:
    saved_run = wf.load_run(saved_run_directory)
    saved_posterior = wf.load_results(saved_run)
    print("Loaded:", saved_run.directory)
    _ = wf.show_results(saved_run)
else:
    print("Set saved_run_directory to inspect an existing run without fitting.")